# LeetCode #47 — Permutations II: hoán vị khi có phần tử trùng

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def permuteUnique(self, nums: List[int]) -> List[List[int]]:
```

Bài #46 sinh hoán vị của các phần tử **khác nhau**. Khi có phần tử **trùng**, cách cũ sinh ra cùng một hoán vị nhiều lần (ví dụ `[1, 1, 2]` có `3! = 6` cách sắp xếp chỉ số nhưng chỉ `3` hoán vị khác nhau).

Mục tiêu của notebook:

1. Hiểu **vì sao** trùng lặp xuất hiện.
2. Quy tắc bỏ trùng với `used`: **`nums[i] == nums[i-1] and not used[i-1]` → bỏ qua** — và ý nghĩa của nó.
3. Cách thứ hai trực quan hơn: backtracking trên **bảng đếm (Counter)** các giá trị.

# 1. Bài toán là gì?

Cho mảng `nums` **có thể chứa phần tử trùng**. Trả về mọi hoán vị **không trùng nhau**, thứ tự tùy ý.

**Constraints (theo đề):**

- `1 <= nums.length <= 8`
- `-10 <= nums[i] <= 10`

---

## Ví dụ

```text
[1, 1, 2]  ->  [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
[1, 2, 3]  ->  6 hoán vị (giống #46)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Hoán vị, tổ hợp, tập con và thứ tự từ điển
3. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [6]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [7]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.3 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [8]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [9]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 315.77 ms | trên set: 0.076 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [10]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [11]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.4 Áp dụng vào bài này

Hoán vị khi có **phần tử trùng**: sắp xếp rồi bỏ qua `nums[i]` nếu nó bằng `nums[i−1]` và `nums[i−1]` **chưa** được dùng (để các bản trùng luôn được chọn theo một thứ tự cố định). Cách khác: quay lui trên **bảng đếm** (`Counter`) các giá trị khác nhau.

# 3. Dùng lại code #46 thì sao?

In [12]:
from typing import List

def permute_46(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

print(permute_46([1, 1, 2]))

[[1, 1, 2], [1, 2, 1], [1, 1, 2], [1, 2, 1], [2, 1, 1], [2, 1, 1]]


`[1, 1, 2]` xuất hiện **2 lần**: một lần chọn số `1` ở chỉ số 0 trước, một lần chọn số `1` ở chỉ số 1 trước. Với số trùng nhau, **hoán đổi vai trò** của chúng không tạo ra hoán vị mới.

Cách vá đơn giản: bỏ trùng bằng `set` các tuple. Nhưng với `[1] * 8`, ta đi qua `8! = 40320` nhánh chỉ để thu được **1** hoán vị.

In [13]:
from itertools import permutations
print("Số nhánh với [1]*8:", len(list(permutations([1] * 8))), "-> số hoán vị khác nhau:", len(set(permutations([1] * 8))))

Số nhánh với [1]*8: 40320 -> số hoán vị khác nhau: 1


# 4. Quy tắc bỏ trùng: ép thứ tự dùng các bản sao

Sau khi **sắp xếp**, các bản sao của cùng một giá trị nằm cạnh nhau: `[1a, 1b, 2]`.

**Quy ước:** các bản sao của cùng một giá trị phải được dùng **theo thứ tự chỉ số** — `1a` luôn được đặt vào `path` **trước** `1b`.

Như vậy, mỗi hoán vị (theo giá trị) chỉ có **một** cách sinh: `[1, 1, 2]` chỉ sinh bằng `1a, 1b, 2`, không bao giờ bằng `1b, 1a, 2`.

Cài đặt: khi muốn chọn `nums[i]` mà `nums[i] == nums[i - 1]` (bản sao kề trước) **chưa được dùng** → đang định dùng `1b` trước `1a` → **bỏ qua**.

```python
if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
    continue
```

In [14]:
def permute_unique(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue                      # bản sao phía trước chưa dùng -> bỏ
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

print(permute_unique([1, 1, 2]))
print(len(permute_unique([1, 2, 3])))
print(permute_unique([1] * 8))

[[1, 1, 2], [1, 2, 1], [2, 1, 1]]
6
[[1, 1, 1, 1, 1, 1, 1, 1]]


## Vì sao là `not used[i - 1]` chứ không phải `used[i - 1]`?

- `used[i - 1] == True`: bản sao phía trước **đang nằm trong `path`** (ở vị trí sớm hơn). Chọn `nums[i]` lúc này là dùng bản sao **theo đúng thứ tự** → **được phép**. Đây là cách sinh `[1, 1, ...]`.
- `used[i - 1] == False`: bản sao phía trước **chưa dùng** (hoặc vừa được gỡ ra khi quay lui). Chọn `nums[i]` lúc này là "vượt mặt" nó → **trùng** với nhánh đã đi qua → **bỏ**.

(Điều kiện `used[i - 1]` (không có `not`) cũng cho ra kết quả đúng — vì nó ép thứ tự **ngược lại** — nhưng cắt tỉa kém hơn nhiều, đi vào rất nhiều nhánh cụt. Xem phần "Lỗi" bên dưới.)

# 5. Mô phỏng: thấy các nhánh bị bỏ

In [15]:
def trace_permute_unique(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result, path, used = [], [], [False] * n
    print("Sau sắp xếp:", nums)

    def backtrack():
        indent = "  " * len(path)
        if len(path) == n:
            print(f"{indent}✓ {path}")
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                print(f"{indent}bỏ nums[{i}]={nums[i]}: bản sao nums[{i - 1}] chưa dùng")
                continue
            used[i] = True
            path.append(nums[i])
            print(f"{indent}chọn nums[{i}]={nums[i]} -> {path}")
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

_ = trace_permute_unique([1, 1, 2])

Sau sắp xếp: [1, 1, 2]
chọn nums[0]=1 -> [1]
  chọn nums[1]=1 -> [1, 1]
    chọn nums[2]=2 -> [1, 1, 2]
      ✓ [1, 1, 2]
  chọn nums[2]=2 -> [1, 2]
    chọn nums[1]=1 -> [1, 2, 1]
      ✓ [1, 2, 1]
bỏ nums[1]=1: bản sao nums[0] chưa dùng
chọn nums[2]=2 -> [2]
  chọn nums[0]=1 -> [2, 1]
    chọn nums[1]=1 -> [2, 1, 1]
      ✓ [2, 1, 1]
  bỏ nums[1]=1: bản sao nums[0] chưa dùng


# 6. Cách 2: backtracking trên bảng đếm

Thay vì làm việc với **chỉ số**, làm việc với **giá trị**: đếm số lần mỗi giá trị xuất hiện. Ở mỗi vị trí, chọn một **giá trị** (không phải chỉ số) còn số lượng `> 0`.

Vì mỗi giá trị chỉ được thử **một lần** ở mỗi tầng, trùng lặp không thể xảy ra — không cần quy tắc tinh tế nào.

In [16]:
from collections import Counter

def permute_unique_counter(nums: List[int]) -> List[List[int]]:
    count = Counter(nums)
    n = len(nums)
    result, path = [], []

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for value in count:
            if count[value] == 0:
                continue
            count[value] -= 1
            path.append(value)
            backtrack()
            path.pop()
            count[value] += 1

    backtrack()
    return result

print(permute_unique_counter([1, 1, 2]))
print(permute_unique_counter([3, 3, 0, 3]))

[[1, 1, 2], [1, 2, 1], [2, 1, 1]]
[[3, 3, 3, 0], [3, 3, 0, 3], [3, 0, 3, 3], [0, 3, 3, 3]]


Hai cách tương đương về độ phức tạp. Cách Counter dễ hiểu hơn; cách `used` là mẫu chuẩn hay gặp trong phỏng vấn và dùng lại được cho nhiều bài.

# 7. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def permuteUnique(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        n = len(nums)
        result, path, used = [], [], [False] * n

        def backtrack():
            if len(path) == n:
                result.append(path[:])
                return
            for i in range(n):
                if used[i]:
                    continue
                if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                    continue
                used[i] = True
                path.append(nums[i])
                backtrack()
                path.pop()
                used[i] = False

        backtrack()
        return result

In [18]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    assert len(actual) == len({tuple(p) for p in actual}), (name, "có trùng")
    print(f"{name:<22} PASS  ({len(actual)} hoán vị)")


sol = Solution()
check("Ví dụ 1", sol.permuteUnique([1, 1, 2]), [[1, 1, 2], [1, 2, 1], [2, 1, 1]])
check("Ví dụ 2", sol.permuteUnique([1, 2, 3]), [list(p) for p in permutations([1, 2, 3])])
check("Toàn giống nhau", sol.permuteUnique([5, 5, 5]), [[5, 5, 5]])
check("Hai cặp trùng", sol.permuteUnique([1, 1, 2, 2]), [list(p) for p in set(permutations([1, 1, 2, 2]))])

import random
random.seed(47)
for _ in range(500):
    nums = [random.randint(-2, 2) for _ in range(random.randint(1, 7))]
    expected = sorted(list(p) for p in set(permutations(nums)))
    assert sorted(sol.permuteUnique(nums[:])) == expected, nums
    assert sorted(permute_unique_counter(nums)) == expected, nums
print("500 test ngẫu nhiên: khớp set(itertools.permutations)")

Ví dụ 1                PASS  (3 hoán vị)
Ví dụ 2                PASS  (6 hoán vị)
Toàn giống nhau        PASS  (1 hoán vị)
Hai cặp trùng          PASS  (6 hoán vị)


500 test ngẫu nhiên: khớp set(itertools.permutations)


# 8. Vì sao đúng? Độ phức tạp?

**Không trùng:** mọi đường đi được chấp nhận đều dùng các bản sao của mỗi giá trị **theo thứ tự chỉ số tăng dần** (bản sau chỉ được chọn khi bản trước đã nằm trong `path`). Mỗi hoán vị theo giá trị có **đúng một** cách gán chỉ số thỏa điều kiện này → sinh đúng một lần.

**Không bỏ sót:** cách gán "theo thứ tự" đó luôn tồn tại cho mọi hoán vị và không vi phạm điều kiện nào → nằm trong cây.

**Số kết quả:** công thức hoán vị lặp `n! / (c₁! · c₂! · …)` với `cᵢ` là số lần xuất hiện của mỗi giá trị.

In [19]:
from math import factorial, prod

nums = [1, 1, 2, 2, 2, 3]
c = Counter(nums)
formula = factorial(len(nums)) // prod(factorial(v) for v in c.values())
print(f"{len(nums)}! / ({' * '.join(f'{v}!' for v in c.values())}) = {formula}; thực tế: {len(sol.permuteUnique(nums))}")

6! / (2! * 3! * 1!) = 60; thực tế: 60


```text
Time:  O(n · n!)  trường hợp xấu (mọi phần tử khác nhau); nhỏ hơn nhiều khi có trùng
Space: O(n) cho path, used, đệ quy (không tính kết quả)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên sắp xếp

Điều kiện so `nums[i]` với `nums[i - 1]` chỉ phát hiện trùng khi các bản sao **nằm cạnh nhau**.

In [20]:
def wrong_unsorted(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i] or (i > 0 and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True; path.append(nums[i])
            backtrack()
            path.pop(); used[i] = False

    backtrack()                                     # thiếu sort
    return result

res = wrong_unsorted([1, 2, 1])
print("Sai :", res, "-> có trùng?", len(res) != len({tuple(p) for p in res}))

Sai : [[1, 2, 1], [1, 1, 2], [2, 1, 1], [2, 1, 1], [1, 1, 2], [1, 2, 1]] -> có trùng? True


## Lỗi 2: Dùng `used[i - 1]` (không có `not`) — đúng nhưng chậm

Vẫn cho kết quả đúng (ép thứ tự ngược: bản sau trước bản trước), nhưng cắt tỉa **muộn**: nhiều nhánh đi sâu rồi mới chết. Đếm số lần gọi đệ quy:

In [21]:
def count_calls(nums: List[int], use_not: bool) -> int:
    nums = sorted(nums)
    n = len(nums)
    used = [False] * n
    calls = 0
    path = []

    def backtrack():
        nonlocal calls
        calls += 1
        if len(path) == n:
            return
        for i in range(n):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1]:
                if use_not and not used[i - 1]:
                    continue
                if not use_not and used[i - 1]:
                    continue
            used[i] = True; path.append(nums[i])
            backtrack()
            path.pop(); used[i] = False

    backtrack()
    return calls

nums = [1] * 6 + [2, 2]
print("Số lần gọi với 'not used[i-1]':", count_calls(nums, True))
print("Số lần gọi với 'used[i-1]'    :", count_calls(nums, False))

Số lần gọi với 'not used[i-1]': 119
Số lần gọi với 'used[i-1]'    : 4830


`nonlocal calls` cho phép hàm lồng **gán lại** biến `calls` của hàm ngoài (khác với `.append` chỉ sửa nội dung list).

## Lỗi 3: Kiểm tra trùng bằng `nums[i] in path`

Sai khi có giá trị trùng: `path = [1]` rồi không cho thêm `1` nữa → mất `[1, 1, 2]`.

## Lỗi 4: Dùng `set` của kết quả để bỏ trùng

Đúng nhưng đi qua `n!` nhánh dù chỉ có rất ít hoán vị khác nhau.

# 10. Mẹo nhận diện pattern

**Có phần tử trùng + kết quả không được trùng** trong backtracking:

| Loại | Quy tắc bỏ trùng |
|---|---|
| Tổ hợp / tập con (#40, #90) | sort; `i > start and a[i] == a[i-1]` → bỏ |
| Hoán vị (#47) | sort; `a[i] == a[i-1] and not used[i-1]` → bỏ |
| Mọi loại | backtracking trên **Counter** giá trị |

Bài liên quan: **#46 Permutations**, **#31 Next Permutation** (lặp `next_permutation` từ mảng đã sắp xếp cũng sinh đủ hoán vị không trùng!), **#996 Number of Squareful Arrays**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[2, 2, 1, 1]`, có bao nhiêu hoán vị khác nhau? Liệt kê theo thứ tự lời giải chính sinh ra.

## Bài 2 — Đếm không sinh

Tính số hoán vị khác nhau bằng công thức `n! / (c₁! c₂! …)`.

```python
count_unique_perms([1, 1, 2])        -> 3
count_unique_perms([1, 2, 3, 4])     -> 24
count_unique_perms([7] * 10)         -> 1
```

In [22]:
def exercise_2_count_unique_perms(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_2_count_unique_perms([1, 1, 2]) == 3
# assert exercise_2_count_unique_perms([1, 2, 3, 4]) == 24
# assert exercise_2_count_unique_perms([7] * 10) == 1

## Bài 3 — Điền chỗ trống

In [23]:
def exercise_3_permute_unique(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            # TODO 1: bỏ nếu used[i]
            # TODO 2: bỏ nếu là bản sao mà bản phía trước chưa dùng
            # TODO 3: chọn, đệ quy, bỏ chọn
            pass

    backtrack()
    return result


# assert sorted(exercise_3_permute_unique([1, 1, 2])) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
# assert exercise_3_permute_unique([0, 0]) == [[0, 0]]

## Bài 4 — Biến thể: hoán vị không trùng của chuỗi

```python
unique_string_perms("aab") -> ["aab", "aba", "baa"]
```

In [24]:
def exercise_4_unique_string_perms(s: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_4_unique_string_perms("aab")) == ["aab", "aba", "baa"]
# assert len(exercise_4_unique_string_perms("aabb")) == 6

## Bài 5 — Mini challenge: sinh bằng Next Permutation

Sinh mọi hoán vị không trùng bằng cách: sắp xếp, rồi lặp lại thuật toán **next permutation** (bài #31) cho tới khi quay về mảng đã sắp xếp. Kết quả phải theo **thứ tự từ điển**.

```python
all_by_next([2, 1, 1]) -> [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
```

In [25]:
def exercise_5_all_by_next(nums: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_all_by_next([2, 1, 1]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
# assert exercise_5_all_by_next([1, 2, 3]) == [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]

# 12. Gợi ý

**Bài 1**
- Mức 1: `4! / (2! · 2!) = 6`.

**Bài 2**
- Mức 1: `Counter(nums).values()` cho các `cᵢ`.

**Bài 3**
- Mức 1: so với code mục 7.

**Bài 4**
- Mức 1: `["".join(p) for p in permute_unique(list(s))]`.

**Bài 5**
- Mức 1: viết lại hàm `next_permutation` tại chỗ từ bài #31.
- Mức 2: lặp: thêm bản sao `nums[:]` vào kết quả, gọi `next_permutation`, dừng khi `nums` quay về `sorted`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
res = permute_unique([2, 2, 1, 1])
print(len(res), res)

6 [[1, 1, 2, 2], [1, 2, 1, 2], [1, 2, 2, 1], [2, 1, 1, 2], [2, 1, 2, 1], [2, 2, 1, 1]]


## Đáp án Bài 2

In [27]:
def answer_2_count_unique_perms(nums: List[int]) -> int:
    c = Counter(nums)
    return factorial(len(nums)) // prod(factorial(v) for v in c.values())

assert answer_2_count_unique_perms([1, 1, 2]) == 3
assert answer_2_count_unique_perms([1, 2, 3, 4]) == 24
assert answer_2_count_unique_perms([7] * 10) == 1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [28]:
def answer_3_permute_unique(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result, path, used = [], [], [False] * n

    def backtrack():
        if len(path) == n:
            result.append(path[:])
            return
        for i in range(n):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return result

assert sorted(answer_3_permute_unique([1, 1, 2])) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert answer_3_permute_unique([0, 0]) == [[0, 0]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_unique_string_perms(s: str) -> List[str]:
    return ["".join(p) for p in permute_unique(list(s))]

assert sorted(answer_4_unique_string_perms("aab")) == ["aab", "aba", "baa"]
assert len(answer_4_unique_string_perms("aabb")) == 6
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def next_permutation(nums: List[int]) -> None:
    n = len(nums)
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1:] = reversed(nums[i + 1:])


def answer_5_all_by_next(nums: List[int]) -> List[List[int]]:
    cur = sorted(nums)
    start = cur[:]
    result = []
    while True:
        result.append(cur[:])
        next_permutation(cur)
        if cur == start:
            return result

assert answer_5_all_by_next([2, 1, 1]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert answer_5_all_by_next([1, 2, 3]) == [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
for _ in range(300):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 6))]
    assert answer_5_all_by_next(nums) == sorted(sol.permuteUnique(nums[:]))
print("Bài 5 PASS (khớp lời giải chính, đúng thứ tự từ điển)")

Bài 5 PASS (khớp lời giải chính, đúng thứ tự từ điển)


# 14. Tổng kết

> Có trùng → **sắp xếp** và ép các bản sao được dùng **theo thứ tự chỉ số**: bỏ `nums[i]` nếu `nums[i] == nums[i-1]` và `not used[i-1]`. Hoặc backtracking trên **Counter** của giá trị.

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp?
- [ ] Điều kiện bỏ: `i > 0 and nums[i] == nums[i - 1] and not used[i - 1]`?
- [ ] Gỡ cả `path` và `used` khi quay lui?
- [ ] Đã thử mảng toàn phần tử giống nhau?